
# Mock Data + Multi-Class Dirichlet GP Classification

In [1]:
import sys
from pathlib import Path

project_root = Path(r"c:\Users\floki\PycharmProjects\dash-azure-prototype")
src_root = project_root / "src"
for p in (project_root, src_root):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
artifact_dir = project_root / "ml" / "models" / "artifacts" / "demo_gpc"
artifact_dir.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.datasets import load_wine, load_digits

In [3]:
from ml.scripts.plotting import (
    plot_true_vs_predicted,
    plot_pca_cumulative_variance,
    plot_pca_loadings,
    plot_scaler_hist,
)
from ml.scripts.trainers import MultiTaskDirichletTrainer
from ml.scripts.log import init_ml_logger, log_classification_summary
from worker.models.models import MultiOutputDirichlet
from worker.models.specs import ModelConfig, PreprocessConfig, AuxilaryData
from worker.models.io_utils import ArtifactIO
from worker.torch_utils import get_default_device
from worker.scalers import TanhTransformer, LogTransformer

In [4]:
# load demo data for testing
dataset = load_digits(as_frame=True)

X = dataset.data.copy()

# 10 binary target columns: one per digit class
y = pd.get_dummies(dataset.target, prefix="target").astype(int)

In [5]:
print(f"n observations {X.shape[0]}")

n observations 1797


In [6]:
pd.concat((X, y), axis=1).head()

,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,target_9
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,0,0,0,0,0
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,0,0,1,0,0,0,0,0,0,0
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,0,0,0,1,0,0,0,0,0,0
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0


In [7]:
# train test split
feature_cols = [f"X{i}" for i in range(X.shape[1])]
target_cols = [f"Y{i}" for i in range(y.shape[1])]

X.columns = feature_cols
y.columns = target_cols

X = X[feature_cols].to_numpy()
y = y[target_cols].to_numpy()

train_x, test_x, train_y, test_y = train_test_split(
    X, y, test_size=0.33, random_state=1, shuffle=True, stratify=dataset.target
)

train_x_raw = train_x.copy()
test_x_raw = test_x.copy()
train_y_raw = train_y.copy()
test_y_raw = test_y.copy()

In [8]:
# feature scaling
scaler_x = StandardScaler()
train_x = scaler_x.fit_transform(train_x)
test_x = scaler_x.transform(test_x)

In [9]:
# setup model
prep = PreprocessConfig(scaler_x=scaler_x)
spec = ModelConfig(
    model_type="multitask_dirichlet",
    features=feature_cols,
    targets=target_cols,
)
aux = AuxilaryData(train_x=train_x, train_y=train_y)
model = MultiOutputDirichlet(spec, prep, aux)

In [ ]:
# conduct training
device = get_default_device()
print(f"Running on {str(device)}")
trainer = MultiTaskDirichletTrainer(model, device=device, lr=0.15)
metrics = trainer.train(train_x, train_y, epochs=150)

Running on cuda


Dirichlet GP training:   0%|          | 0/150 [00:00<?, ?it/s]

In [ ]:
# check if storing / loading of artifacts works
ArtifactIO.save(artifact_dir, model=model, spec=spec, prep=prep, aux=aux)
loaded_model = ArtifactIO.load(artifact_dir, device=device)

In [ ]:
# test predictions on test set
preds = loaded_model.predict(test_x_raw, device=device)
pred_cls = preds.get('cls')
pred_prob = preds.get('prob')

preds_train = loaded_model.predict(train_x_raw, device=device)
pred_cls_train = preds_train.get('cls')
pred_prob_train = preds_train.get('prob')

In [ ]:
logger, _ = init_ml_logger(artifact_dir, "training")
if logger is not None:
    logger.info("Train time (s): %.3f", trainer.duration)
    log_classification_summary(
        logger,
        y_true=train_y_raw,
        y_pred=pred_cls_train,
        y_score=pred_prob_train,
        target_cols=target_cols,
        phase="Train",
        average="binary",
    )
    log_classification_summary(
        logger,
        y_true=test_y_raw,
        y_pred=pred_cls,
        y_score=pred_prob,
        target_cols=target_cols,
        phase="Test",
        average="binary",
    )